In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train_df = pd.read_csv('Project Data/lending_club_train.csv')
test_df = pd.read_csv('Project Data/lending_club_test.csv')

In [ ]:
print("--- Train Data Head ---")
display(train_df.head())

print("\n--- Test Data Head ---")
display(test_df.head())

In [ ]:
# Converting "issue_d" to datetime format
train_df['issue_d_dt'] = pd.to_datetime(train_df['issue_d'], errors='coerce')

In [ ]:
# Filtering for 2016 data
train_2016 = train_df[train_df['issue_d_dt'].dt.year == 2016].copy()
print(f"Original training rows: {train_df.shape[0]}")
print(f"Filtered 2016 training rows: {train_2016.shape[0]}")
print(f"Months included in 2016 train set: {sorted(train_2016['issue_d_dt'].dt.month.unique())}")

In [ ]:
# Doing the same for the test data
test_df['issue_d_dt'] = pd.to_datetime(test_df['issue_d'], errors='coerce')
test_2016 = test_df[test_df['issue_d_dt'].dt.year == 2016].copy()
print(f"Original test rows: {test_df.shape[0]}")
print(f"Filtered 2016 test rows: {test_2016.shape[0]}")
print(f"Months included in 2016 test set: {sorted(test_2016['issue_d_dt'].dt.month.unique())}")

In [ ]:
# Checking for nulls
print("--- TRAIN NULL % ---")
train_nulls = (train_2016.isnull().sum() / len(train_2016) * 100).round(2)
print(train_nulls[train_nulls > 0].sort_values(ascending=False))

print("\n--- TEST NULL % ---")
test_nulls = (test_2016.isnull().sum() / len(test_2016) * 100).round(2)
print(test_nulls[test_nulls > 0].sort_values(ascending=False))

In [ ]:
# Map sub_grade (A1..G5) to an ordinal 1..35 instead of one-hot encoding it
grades = [f"{g}{n}" for g in "ABCDEFG" for n in range(1, 6)]
grade_map = {g: i + 1 for i, g in enumerate(grades)}
train_2016['sub_grade'] = train_2016['sub_grade'].str.strip().map(grade_map)
test_2016['sub_grade'] = test_2016['sub_grade'].str.strip().map(grade_map)

In [ ]:
# 1. Define columns to drop (IDs, dates, and data leakage columns)
cols_to_drop = ['id', 'issue_d', 'issue_d_dt', 'loan_status']

# 2. Separate the target variable (int_rate) from the feature sets
train_y = train_2016['int_rate']
test_y = test_2016['int_rate']

train_features = train_2016.drop(columns=cols_to_drop + ['int_rate'])
test_features = test_2016.drop(columns=cols_to_drop + ['int_rate'])

# # 3. Combine temporarily to ensure One-Hot Encoding creates identical columns for both
# combined_features = pd.concat([train_features, test_features], axis=0)
# combined_encoded = pd.get_dummies(combined_features, drop_first=True)

# 3. One-Hot Encode each set separately (all categories kept, no drop_first)
train_encoded = pd.get_dummies(train_features, dtype=float)
test_encoded = pd.get_dummies(test_features, dtype=float)

# # 4. Split them back apart into train and test sets
# train_encoded = combined_encoded.iloc[:len(train_2016)]
# test_encoded = combined_encoded.iloc[len(train_2016):]

# 4. Align test to the TRAIN columns: categories unseen in train (e.g. home_ownership_ANY)
#    are dropped, and dummies that exist only in train are added as 0
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0.0)

# Quick check on the final shapes
print("Train features shape:", train_encoded.shape)
print("Test features shape:", test_encoded.shape)

In [ ]:
# ---------------- Baseline regression tree (default settings) ----------------
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def report(name, model):
    """Fit on 2016 train, score on both train and 2016 test."""
    model.fit(train_encoded, train_y)
    results = {}
    for label, X, y in [("train", train_encoded, train_y), ("test", test_encoded, test_y)]:
        pred = model.predict(X)
        results[label] = {
            "R2": r2_score(y, pred),
            "RMSE": mean_squared_error(y, pred) ** 0.5,
            "MAE": mean_absolute_error(y, pred),
        }
    print(f"\n--- {name} ---")
    print(pd.DataFrame(results).T.round(4))
    return model

# 1. Reference point: always predict the training mean
report("Mean-predictor reference", DummyRegressor(strategy="mean"))

# 2. Baseline: regression tree under default settings
baseline_tree = report("Baseline regression tree (defaults)", DecisionTreeRegressor(random_state=42))
print("Leaves:", baseline_tree.get_n_leaves(), " Depth:", baseline_tree.get_depth())

# 3. Which features does the tree rely on?
importances = pd.Series(baseline_tree.feature_importances_, index=train_encoded.columns)
print("\nTop 10 feature importances:")
print(importances.sort_values(ascending=False).head(10).round(4))